# 05 Prepare faithfulness evaluation

Validate the frozen baseline and prepare the 618 evaluation pairs.

**Kernel:** `.venv-scoring` (`.venv-scoring/Scripts/python.exe`).


In [1]:
from pathlib import Path
import sys
import os
import json
import pandas as pd
from IPython.display import display

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / 'configs' / 'config.yaml').is_file()
             and (path / 'outputs' / 'baseline.csv').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the csc173-midterm repository.')
os.chdir(ROOT)
if Path(sys.prefix).resolve() != (ROOT / '.venv-scoring').resolve():
    raise RuntimeError('Select the .venv-scoring notebook kernel, then restart.')
import hashlib

RUN = ROOT / 'outputs' / 'faithfulness' / 'run_01'
N = 618

print('Run folder:', RUN)
print('Kernel:', sys.executable)


Run folder: c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\outputs\faithfulness\run_01
Kernel: c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\.venv-scoring\Scripts\python.exe


## Function definitions


### Shared helper functions

Reading, validation, file saving, and resumption helpers.


In [2]:
def read_csv(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')


def require(condition, message):
    if not condition:
        raise ValueError(message)


def nonblank(frame, columns):
    for column in columns:
        require(column in frame, f'Missing column: {column}')
        require(frame[column].map(lambda x: isinstance(x, str) and bool(x.strip())).all(),
                f'Blank or invalid text in {column}')


def reference_list(text):
    values = json.loads(text)
    require(isinstance(values, list) and len(values) > 0, 'Missing reference list')
    require(all(isinstance(x, str) and bool(x.strip()) for x in values), 'Invalid reference')
    return values


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False, lineterminator='\n')
    os.replace(temporary, path)


def load_eval():
    frame = read_csv(RUN / 'evaluation_pairs.csv')
    nonblank(frame, ['sample_id', 'abstract', 'baseline_summary', 'human_tldrs'])
    require(len(frame) == N and frame.sample_id.is_unique, 'Expected 618 unique papers')
    frame.human_tldrs.map(reference_list)
    return frame


### `prepare_evaluation()`

Prepare the evaluation table from the original baseline CSV.

#### Purpose and inputs

This function reads `outputs/baseline.csv` and creates a common table for the next stages.

The table is read and verifies the following:
1. Required columns are not empty
2. Source columns types are preserved
3. There are 618 unique IDs
4. There is a usable JSON list

This function then proceeds to rename derived columns to avoid redundancy
- `id` --> `sample_id`
- `summary` --> `baseline_summary`

#### Output and interpretation

The output is `RUN / 'evaluation_pairs.csv'`, with 618 rows and columns `sample_id`, `abstract`, `baseline_summary`, and `human_tldrs`. 


In [3]:
def prepare_evaluation():
    """Create the evaluation table from baseline.csv without needing split files."""
    import json
    import pandas as pd

    source = ROOT / 'outputs' / 'baseline.csv'
    frame = read_csv(source)
    nonblank(frame, ['id', 'abstract', 'summary', 'human_tldrs'])
    require(len(frame) == N and frame.id.is_unique, 'Expected 618 rows and unique IDs')
    frame.human_tldrs.map(reference_list)
    frame = frame.rename(columns={'id': 'sample_id', 'summary': 'baseline_summary'})
    frame = frame[['sample_id', 'abstract', 'baseline_summary', 'human_tldrs']]

    destination = RUN / 'evaluation_pairs.csv'
    if destination.exists():
        require(read_csv(destination).equals(frame), 'Derived input changed')
    else:
        atomic_csv(frame, destination)
    print('Validated', len(frame), 'papers:', destination)


## Execution and results


In [4]:
prepare_evaluation()
evaluation = load_eval()
print('Papers:', len(evaluation), '| Unique IDs:', evaluation.sample_id.nunique())
display(evaluation[['sample_id', 'baseline_summary']].head())


Validated 618 papers: c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\outputs\faithfulness\run_01\evaluation_pairs.csv
Papers: 618 | Unique IDs: 618


,sample_id,baseline_summary
0,SJ1Xmf-Rb,"We propose FearNet, a generative model that do..."
1,S1xzyhR9Y7,Multi-view learning can provide self-supervisi...
2,HJDUjKeA-,We show how discrete objects can be learnt in ...
3,BJgLg3R9KQ,We extend a state-of-the-art attention network...
4,BklpOo09tQ,We show that adversarial examples can be gener...
